In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
from src import features as ft

df = pd.read_parquet(Path("..") / "data" / "processed" / "clean.parquet")
texts = (df["ns_tpa_des_2025"].fillna("") + " " + df["ns_mpa_des_2025"].fillna("")).str.strip()
texts = texts.where(texts != "")
country = df[ft.COUNTRY_SRC].astype("object").fillna("unknown")
print("городов с описанием:", texts.notna().sum(), "из", len(df))

words, pairs = ft.word_bigram_counts(texts)
print("уникальных слов:", len(words), "| уникальных биграмм:", len(pairs))
top_w = words.most_common(30)
top_b = pairs.most_common(30)
t6 = pd.DataFrame({
    "слово": [w for w, _ in top_w[:5]],
    "частота": [n for _, n in top_w[:5]],
    "биграмма": [" ".join(p) for p, _ in top_b[:5]],
    "частота биграммы": [n for _, n in top_b[:5]],
})
display(t6)
print("топ-30 слов:", [w for w, _ in top_w])
print("топ-30 биграмм:", [" ".join(p) for p, _ in top_b])

has = texts.notna()
five = [w for w, _ in top_w[:5]]
share = ft.word_share_by_group(texts[has], country[has], five)
big3 = country[has].value_counts().head(3).index.tolist()
display(share.loc[big3].round(3))

tfidf = ft.tfidf_top_words(texts[has], country[has], big3, top=5)
for group, items in tfidf.items():
    print(group)
    for word, inside, overall in items:
        print(f"   {word:<18} в стране {inside:.3f} | в наборе {overall:.3f}")

городов с описанием: 383 из 1089
уникальных слов: 127 | уникальных биграмм: 368


,слово,частота,биграмма,частота биграммы
0,area,315,protection area,169
1,park,229,environmental protection,158
2,protection,181,area environmental,53
3,environmental,158,national park,52
4,reserve,125,ramsar site,41


топ-30 слов: ['area', 'park', 'protection', 'environmental', 'reserve', 'ecological', 'national', 'conservation', 'zone', 'interest', 'site', 'natural', 'wetland', 'ramsar', 'international', 'importance', 'relevant', 'wildlife', 'heritage', 'protective', 'nature', 'refuge', 'protected', 'critical', 'treatment', 'priority', 'acpt', 'landscape', 'state', 'biosphere']
топ-30 биграмм: ['protection area', 'environmental protection', 'area environmental', 'national park', 'ramsar site', 'site wetland', 'wetland international', 'international importance', 'area relevant', 'relevant ecological', 'ecological interest', 'ecological conservation', 'wildlife refuge', 'park park', 'interest area', 'protective zone', 'critical area', 'area treatment', 'treatment priority', 'priority acpt', 'biosphere reserve', 'park environmental', 'area park', 'unesco mab', 'mab biosphere', 'cultural heritage', 'heritage landscape', 'nature reserve', 'protected area', 'state park']


,area,park,protection,environmental,reserve
Brazil,0.617,0.492,0.578,0.578,0.148
México,0.397,0.500,0.250,0.000,0.235
Venezuela,0.692,0.212,0.038,0.019,0.212


Brazil
   environmental      в стране 0.313 | в наборе 0.106
   protection         в стране 0.283 | в наборе 0.108
   park               в стране 0.326 | в наборе 0.177
   area               в стране 0.249 | в наборе 0.133
   relevant           в стране 0.044 | в наборе 0.015
México
   conservation       в стране 0.170 | в наборе 0.036
   ecological         в стране 0.166 | в наборе 0.064
   state              в стране 0.121 | в наборе 0.022
   site               в стране 0.098 | в наборе 0.035
   importance         в стране 0.090 | в наборе 0.034
Venezuela
   treatment          в стране 0.226 | в наборе 0.031
   priority           в стране 0.226 | в наборе 0.031
   acpt               в стране 0.226 | в наборе 0.031
   critical           в стране 0.226 | в наборе 0.031
   protective         в стране 0.198 | в наборе 0.030
